# MatForge SR — Notebook 01: Fine-tuning pipeline
#
# Fine-tunes a Real-ESRGAN+ generator (RRDBNet, 6 RRDB blocks, ×4 scale) on
# the MatSynth RGB subset to specialise super-resolution for PBR material
# surface imagery.
#
# Execution modes (set DRY_RUN in Cell 2):
# - DRY_RUN = True  → 3 epochs Phase 1 + 2 epochs Phase 2, validate every
#                     epoch, verbose diagnostics. No useful checkpoints saved.
# - DRY_RUN = False → 30 epochs Phase 1 + 20 epochs Phase 2, validate every
#                     5 epochs, clean output. Checkpoints saved to /kaggle/working.
#
# Run cells in order. The training loop is resumable from any checkpoint.

# Cell 1 — Package installation and weight download

In [1]:
import subprocess, urllib.request, hashlib
from pathlib import Path
 
subprocess.run(["pip", "install", "lpips", "--quiet"], check=True)
 
# ---------------------------------------------------------------------------
# Download Real-ESRGAN+ generator weights (6 RRDB blocks) from the Kaggle
# dataset created for this project. The path below assumes the dataset is
# attached as input with name "matforge-sr-weights".
#
# On the very first run the dataset does not exist yet. In that case, the
# block below downloads the weights directly from the official GitHub release
# and saves them to /kaggle/working/ so they can be uploaded as the seed
# dataset.
# ---------------------------------------------------------------------------
 
SR_WEIGHTS_KAGGLE   = Path("/kaggle/input/datasets/mjgut05/matforge-sr-weights/RealESRGAN_x4plus.pth")
SR_WEIGHTS_FALLBACK = Path("/kaggle/working/RealESRGAN_x4plus.pth")
 
OFFICIAL_URL = (
    "https://github.com/xinntao/Real-ESRGAN/releases/download/"
    "v0.1.0/RealESRGAN_x4plus.pth"
)
 
if SR_WEIGHTS_KAGGLE.exists():
    PRETRAINED_PATH = SR_WEIGHTS_KAGGLE
    print(f"[OK] Weights loaded from Kaggle dataset: {PRETRAINED_PATH}")
elif SR_WEIGHTS_FALLBACK.exists():
    PRETRAINED_PATH = SR_WEIGHTS_FALLBACK
    print(f"[OK] Weights loaded from working dir fallback: {PRETRAINED_PATH}")
else:
    print("Weights not found in dataset — downloading from GitHub release...")
    urllib.request.urlretrieve(OFFICIAL_URL, SR_WEIGHTS_FALLBACK)
    PRETRAINED_PATH = SR_WEIGHTS_FALLBACK
    print(f"[OK] Downloaded to {PRETRAINED_PATH}  "
          f"({PRETRAINED_PATH.stat().st_size / 1e6:.1f} MB)")
    print("ACTION REQUIRED: upload this file to the 'matforge-sr-weights' "
          "Kaggle dataset so future runs skip the download.")
 
subprocess.run(["pip", "install", "lpips", "--quiet"], check=True)
print("lpips ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.6 MB/s eta 0:00:00
[OK] Weights loaded from Kaggle dataset: /kaggle/input/datasets/mjgut05/matforge-sr-weights/RealESRGAN_x4plus.pth
lpips ready.


# Cell 2 — Imports and configuration

In [2]:
import os, json, math, time, random, io
from pathlib import Path
from collections import defaultdict
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler
from torchvision.transforms import functional as TF
import torchvision.models as tvm
import lpips as lpips_lib
 
# ── Execution mode ────────────────────────────────────────────────────────────
DRY_RUN = False   # set False for the full training run
 
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
 
# ── Dataset paths ─────────────────────────────────────────────────────────────
DATASET_ROOT = Path(
    "/kaggle/input/datasets/mjgut05/matforge-pbr-dataset/matforge_dataset"
)
RGB_DIR    = DATASET_ROOT / "maps" / "rgb"
SPLIT_CSV  = (
    Path("/kaggle/input/datasets/mjgut05/matforge-pbr-dataset")
    / "matforge_split.csv"
)
COL_FILENAME = "filename"
COL_GROUP    = "functional_group"
 
# ── Output paths ──────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("/kaggle/working")
CKPT_DIR   = OUTPUT_DIR / "sr_checkpoints"
PANELS_DIR = OUTPUT_DIR / "sr_panels"
CKPT_DIR.mkdir(exist_ok=True)
PANELS_DIR.mkdir(exist_ok=True)
 
# ── Phase 1 hyperparameters (generator only) ──────────────────────────────────
PHASE1_EPOCHS    = 3  if DRY_RUN else 30
LR_G_PHASE1      = 5e-5
FREEZE_BLOCKS    = 2          # freeze first N RRDB blocks for first 5 epochs
UNFREEZE_EPOCH   = 1 if DRY_RUN else 5
WEIGHT_DECAY_G   = 1e-4
 
# ── Phase 2 hyperparameters (generator + discriminator) ───────────────────────
PHASE2_EPOCHS    = 2  if DRY_RUN else 20
LR_G_PHASE2      = 1e-5
LR_D             = 1e-4
WEIGHT_DECAY_D   = 0.0        # no weight decay on discriminator (standard GAN practice)
R1_INTERVAL      = 16         # lazy R1: compute gradient penalty every N steps
R1_LAMBDA        = 10.0
W_FM             = 10.0       # feature matching loss weight
 
# Progressive adversarial weight: (epoch_start_inclusive, epoch_end_exclusive) → weight
GAN_ADV_SCHED = {(0, 5): 0.01, (5, 10): 0.05, (10, 999): 0.10}
 
def get_adv_weight(epoch_gan: int) -> float:
    for (s, e), w in GAN_ADV_SCHED.items():
        if s <= epoch_gan < e:
            return w
    return 0.10
 
# ── Loss weights (Phase 1 and 2) ──────────────────────────────────────────────
W_L1   = 1.0
W_PERC = 1.0
W_LPIPS = 0.5
 
# ── Training logistics ────────────────────────────────────────────────────────
BATCH_SIZE      = 8
NUM_WORKERS     = 2
GRAD_CLIP       = 1.0
VALIDATE_EVERY  = 1 if DRY_RUN else 5
SAVE_CKPT_EVERY = 10           # save periodic checkpoint every N epochs
EARLY_STOP_PATIENCE = 3       # validation rounds without LPIPS improvement
 
# ── SR model configuration ────────────────────────────────────────────────────
SCALE        = 4
TILE_SIZE    = 256            # input tile size → output 1024×1024
PATCH_SIZE   = 256            # crop size for training pairs (HQ side)
LQ_SIZE      = PATCH_SIZE // SCALE  # 64×64
 
# ── Degradation parameters ────────────────────────────────────────────────────
NOISE_SIGMA_MAX  = 10.0       # Gaussian noise std, drawn from U(0, NOISE_SIGMA_MAX)
BLUR_SIGMA_MIN   = 0.2        # Gaussian blur std range
BLUR_SIGMA_MAX   = 1.5
JPEG_QUALITY_MIN = 70
JPEG_QUALITY_MAX = 95
 
# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
 
print(f"Mode   : {'DRY RUN' if DRY_RUN else 'FULL TRAINING'}")
print(f"Phase 1: {PHASE1_EPOCHS} epochs")
print(f"Phase 2: {PHASE2_EPOCHS} epochs")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mode   : FULL TRAINING
Phase 1: 30 epochs
Phase 2: 20 epochs
Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


# Cell 3 — Generator architecture: RRDBNet (6 RRDB blocks, ×4)
#
# Self-contained implementation that mirrors the official Real-ESRGAN
# architecture exactly, avoiding the basicsr/torchvision compatibility issue
# with torchvision >= 0.17 (removed functional_tensor module).
# The weight keys produced by this implementation match those in the official
# checkpoint, so load_state_dict works without key remapping.

In [3]:
class ResidualDenseBlock(nn.Module):
    """
    Residual Dense Block — the atomic unit of RRDB.
    Five convolutions with dense connections; residual scaling factor 0.2
    matches the original ESRGAN implementation for gradient stability.
    """
    def __init__(self, num_feat: int = 64, num_grow_ch: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(num_feat,               num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat +   num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2*num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3*num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4*num_grow_ch, num_feat,    3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
        # Weight initialisation: scale down to 0.1 of default for stable training
        for m in [self.conv1, self.conv2, self.conv3, self.conv4, self.conv5]:
            nn.init.kaiming_normal_(m.weight, a=0.2)
            m.weight.data *= 0.1
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x
 
 
class RRDB(nn.Module):
    """Residual-in-Residual Dense Block: three RDBs with residual scaling."""
    def __init__(self, num_feat: int = 64, num_grow_ch: int = 32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.rdb3(self.rdb2(self.rdb1(x)))
        return out * 0.2 + x
 
 
class RRDBNet(nn.Module):
    """
    Generator backbone used in Real-ESRGAN / Real-ESRGAN+.
 
    Plan A uses num_block=6 (Real-ESRGAN+, ~4.5M params).
    Plan B uses num_block=23 (Real-ESRGAN, ~16.7M params).
 
    Upsampling uses nearest-neighbour interpolation followed by a learned
    convolution at each ×2 stage, totalling ×4 for scale=4.
    """
    def __init__(
        self,
        num_in_ch: int  = 3,
        num_out_ch: int = 3,
        num_feat: int   = 64,
        num_block: int  = 6,
        num_grow_ch: int = 32,
        scale: int      = 4,
    ):
        super().__init__()
        self.scale = scale
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)
        self.body       = nn.Sequential(*[RRDB(num_feat, num_grow_ch)
                                          for _ in range(num_block)])
        self.conv_body  = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up1   = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2   = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr    = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last  = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
        self.lrelu      = nn.LeakyReLU(negative_slope=0.2, inplace=True)
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat      = self.conv_first(x)
        body_feat = self.conv_body(self.body(feat))
        feat      = feat + body_feat
        feat = self.lrelu(self.conv_up1(
            F.interpolate(feat, scale_factor=2, mode="nearest")))
        feat = self.lrelu(self.conv_up2(
            F.interpolate(feat, scale_factor=2, mode="nearest")))
        return self.conv_last(self.lrelu(self.conv_hr(feat)))
 
    def freeze_blocks(self, n: int) -> None:
        """Freeze the first n RRDB blocks. n=0 unfreezes everything."""
        for i, block in enumerate(self.body):
            requires_grad = (i >= n)
            for p in block.parameters():
                p.requires_grad = requires_grad
 
    def get_intermediate_features(self, x: torch.Tensor) -> list:
        """
        Returns feature maps from each RRDB block.
        Used by the feature matching loss in Phase 2.
        """
        feats = []
        feat = self.conv_first(x)
        for block in self.body:
            feat = block(feat)
            feats.append(feat)
        return feats
 
 
def build_generator(pretrained_path: Path, num_block: int = 6) -> RRDBNet:
    """
    Instantiates RRDBNet and loads pretrained weights.
    The official Real-ESRGAN+ checkpoint stores weights under a 'params_ema'
    or 'params' key; this function handles both layouts automatically.
    """
    net = RRDBNet(num_block=num_block).to(DEVICE)
 
    ckpt = torch.load(pretrained_path, map_location="cpu")
    # Official checkpoints wrap weights under 'params_ema' or 'params'
    if "params_ema" in ckpt:
        state = ckpt["params_ema"]
    elif "params" in ckpt:
        state = ckpt["params"]
    else:
        state = ckpt   # raw state dict
 
    missing, unexpected = net.load_state_dict(state, strict=False)
    if missing:
        print(f"  [WARN] Missing keys ({len(missing)}): {missing[:3]} ...")
    if unexpected:
        print(f"  [WARN] Unexpected keys ({len(unexpected)}): {unexpected[:3]} ...")
 
    params = sum(p.numel() for p in net.parameters()) / 1e6
    print(f"[OK] Generator loaded  |  blocks={num_block}  |  params={params:.2f}M")
    return net
 
 
generator = build_generator(PRETRAINED_PATH, num_block=23)

[OK] Generator loaded  |  blocks=23  |  params=16.70M


# Cell 4 — Discriminator architecture: U-Net PatchGAN with spectral norm
#
# Conditional discriminator that receives both the LQ input (upsampled to HQ
# resolution for alignment) and the HQ/SR image concatenated as input.
# Spectral normalisation on all Conv layers stabilises training against the
# already-mature generator.

In [4]:
def spectral_norm(m: nn.Module) -> nn.Module:
    """Applies spectral normalisation to all Conv2d layers in a module."""
    for name, layer in m.named_modules():
        if isinstance(layer, nn.Conv2d):
            nn.utils.spectral_norm(layer)
    return m
 
 
class UNetDiscriminator(nn.Module):
    """
    U-Net discriminator from the Real-ESRGAN paper.
    Input: concatenation of LQ (upsampled) and HQ/SR → 6 channels.
    Output: spatial map of real/fake logits at 1/8 of input resolution.
 
    The U-Net structure (encoder + decoder with skip connections) lets the
    discriminator attend to both global structure (bottleneck) and local
    texture detail (skip connections), which is important for PBR material
    textures that have both global pattern and fine-grained detail.
    """
    def __init__(self, num_in_ch: int = 6, num_feat: int = 64):
        super().__init__()
        # Encoder
        self.conv0  = nn.Conv2d(num_in_ch, num_feat,      3, 1, 1)
        self.conv1  = nn.Conv2d(num_feat,    num_feat*2,  4, 2, 1)
        self.conv2  = nn.Conv2d(num_feat*2,  num_feat*4,  4, 2, 1)
        self.conv3  = nn.Conv2d(num_feat*4,  num_feat*8,  4, 2, 1)
        # Decoder with skip connections
        self.conv4  = nn.Conv2d(num_feat*8,  num_feat*4,  3, 1, 1)
        self.conv5  = nn.Conv2d(num_feat*4*2,num_feat*2,  3, 1, 1)  # skip from conv2
        self.conv6  = nn.Conv2d(num_feat*2*2,num_feat,    3, 1, 1)  # skip from conv1
        self.conv7  = nn.Conv2d(num_feat*2,  num_feat,    3, 1, 1)  # skip from conv0
        self.conv8  = nn.Conv2d(num_feat,    1,           3, 1, 1)
        self.lrelu  = nn.LeakyReLU(0.2, inplace=True)
        spectral_norm(self)
 
    def forward(self, x: torch.Tensor) -> tuple:
        """
        Returns (logits, feature_list).
        feature_list contains intermediate activations for feature matching loss.
        """
        f0 = self.lrelu(self.conv0(x))
        f1 = self.lrelu(self.conv1(f0))
        f2 = self.lrelu(self.conv2(f1))
        f3 = self.lrelu(self.conv3(f2))
        # Decoder
        d3 = self.lrelu(self.conv4(f3))
        d3 = F.interpolate(d3, size=f2.shape[2:], mode="bilinear", align_corners=False)
        d2 = self.lrelu(self.conv5(torch.cat([d3, f2], dim=1)))
        d2 = F.interpolate(d2, size=f1.shape[2:], mode="bilinear", align_corners=False)
        d1 = self.lrelu(self.conv6(torch.cat([d2, f1], dim=1)))
        d1 = F.interpolate(d1, size=f0.shape[2:], mode="bilinear", align_corners=False)
        d0 = self.lrelu(self.conv7(torch.cat([d1, f0], dim=1)))
        out = self.conv8(d0)
        features = [f1, f2, f3, d2, d1]
        return out, features
 
 
def build_discriminator() -> UNetDiscriminator:
    disc = UNetDiscriminator(num_in_ch=6).to(DEVICE)
    params = sum(p.numel() for p in disc.parameters()) / 1e6
    print(f"[OK] Discriminator built  |  params={params:.2f}M")
    return disc

# Cell 5 — SR dataset: LQ-HQ pair generation with synthetic degradation
#
# Loads RGB images at 1K from the MatSynth dataset using the frozen
# matforge_split.csv. Generates LQ counterparts on-the-fly in __getitem__
# via the degradation pipeline (bicubic ×4 + Gaussian noise + Gaussian blur
# + JPEG compression). No PBR maps are loaded; only the RGB channel is used.

In [5]:
import torchvision.transforms.functional as TVF
from torchvision.transforms import RandomCrop
 
class SRDegradationPipeline:
    """
    Generates a degraded LQ version of a HQ image crop.
 
    Pipeline (applied in fixed order):
        1. Bicubic downscale ×4  (256 → 64 px)
        2. Additive Gaussian noise  σ ~ U(0, NOISE_SIGMA_MAX)
        3. Gaussian blur  σ ~ U(BLUR_SIGMA_MIN, BLUR_SIGMA_MAX)
        4. JPEG compression  quality ~ U(JPEG_QUALITY_MIN, JPEG_QUALITY_MAX)
 
    All randomness is controlled by Python's random module, which is seeded
    globally. The fixed order is intentional: it mirrors real-world image
    capture and transmission (optics blur → sensor noise → transmission codec).
 
    Output tensors are float32 in [0, 1].
    """
 
    @staticmethod
    def _gaussian_kernel(sigma: float, kernel_size: int) -> torch.Tensor:
        """1-D Gaussian kernel; used to build a 2-D separable blur filter."""
        x = torch.arange(kernel_size, dtype=torch.float32) - kernel_size // 2
        gauss = torch.exp(-0.5 * (x / sigma) ** 2)
        return gauss / gauss.sum()
 
    @staticmethod
    def _apply_blur(img: torch.Tensor, sigma: float) -> torch.Tensor:
        """Applies separable Gaussian blur in float32."""
        ks = 2 * math.ceil(3 * sigma) + 1
        k1d = SRDegradationPipeline._gaussian_kernel(sigma, ks)
        kernel = k1d.outer(k1d).unsqueeze(0).unsqueeze(0)  # (1,1,ks,ks)
        kernel = kernel.expand(img.shape[0], 1, ks, ks)     # per-channel
        pad = ks // 2
        img_pad = F.pad(img.unsqueeze(0), [pad, pad, pad, pad], mode="reflect")
        blurred = F.conv2d(img_pad, kernel.to(img.device), groups=img.shape[0])
        return blurred.squeeze(0).clamp(0.0, 1.0)
 
    @staticmethod
    def _apply_jpeg(img: torch.Tensor, quality: int) -> torch.Tensor:
        """Applies JPEG compression via PIL round-trip. In-memory, no disk I/O."""
        pil = TVF.to_pil_image(img.clamp(0, 1))
        buf = io.BytesIO()
        pil.save(buf, format="JPEG", quality=quality)
        buf.seek(0)
        return TVF.to_tensor(Image.open(buf).convert("RGB"))
 
    def __call__(self, hq: torch.Tensor) -> torch.Tensor:
        """
        hq: (3, PATCH_SIZE, PATCH_SIZE) float32 tensor in [0, 1].
        Returns lq: (3, LQ_SIZE, LQ_SIZE) float32 tensor in [0, 1].
        """
        # Step 1: bicubic downscale
        lq = F.interpolate(
            hq.unsqueeze(0),
            size=(LQ_SIZE, LQ_SIZE),
            mode="bicubic",
            align_corners=False,
        ).squeeze(0).clamp(0.0, 1.0)
 
        # Step 2: additive Gaussian noise
        sigma = random.uniform(0.0, NOISE_SIGMA_MAX) / 255.0
        if sigma > 0:
            lq = (lq + torch.randn_like(lq) * sigma).clamp(0.0, 1.0)
 
        # Step 3: Gaussian blur
        blur_sigma = random.uniform(BLUR_SIGMA_MIN, BLUR_SIGMA_MAX)
        lq = self._apply_blur(lq, blur_sigma)
 
        # Step 4: JPEG compression
        quality = random.randint(JPEG_QUALITY_MIN, JPEG_QUALITY_MAX)
        lq = self._apply_jpeg(lq, quality)
 
        return lq
 
 
class SRDataset(Dataset):
    """
    Super-resolution dataset built on top of the existing MatSynth RGB split.
 
    Each call to __getitem__ returns:
        hq: (3, PATCH_SIZE, PATCH_SIZE) float32 in [0, 1] — HQ crop from 1K image
        lq: (3, LQ_SIZE,    LQ_SIZE)    float32 in [0, 1] — degraded version of hq
 
    Augmentation (training only):
        - Random horizontal flip (p=0.5)
        - Random vertical flip (p=0.5)
        - Random rotation 0/90/180/270° (p=0.25 each)
    No Normal-map component transforms are needed (RGB only).
 
    The split CSV is the same frozen file used for MatForge training, ensuring
    that the SR validation set contains identical images to the MatForge
    validation set. No new split logic is introduced.
    """
 
    def __init__(self, df: pd.DataFrame, split: str, augment: bool = True):
        assert split in ("train", "val")
        self.df        = df[df["split"] == split].reset_index(drop=True)
        self.split     = split
        self.augment   = augment and (split == "train")
        self.degrade   = SRDegradationPipeline()
 
    def __len__(self) -> int:
        return len(self.df)
 
    @staticmethod
    def _random_crop(img: torch.Tensor, size: int) -> torch.Tensor:
        i, j, h, w = RandomCrop.get_params(img, (size, size))
        return TVF.crop(img, i, j, h, w)
 
    @staticmethod
    def _augment(img: torch.Tensor) -> torch.Tensor:
        if random.random() < 0.5:
            img = TVF.hflip(img)
        if random.random() < 0.5:
            img = TVF.vflip(img)
        k = random.randint(0, 3)
        if k > 0:
            img = torch.rot90(img, k, dims=[1, 2])
        return img
 
    def __getitem__(self, idx: int) -> dict:
        fname = str(self.df.iloc[idx][COL_FILENAME])
        fpath = RGB_DIR / fname
        if not fpath.exists():
            fpath = RGB_DIR / (fname + ".png")
        img = TVF.to_tensor(Image.open(fpath).convert("RGB"))
        # img is float32 in [0, 1], shape (3, H, W) where H=W=1024
 
        hq = self._random_crop(img, PATCH_SIZE)   # (3, 256, 256)
 
        if self.augment:
            hq = self._augment(hq)
 
        lq = self.degrade(hq)                      # (3, 64, 64)
 
        return {"hq": hq, "lq": lq, "filename": fname}
 
 
def build_sr_loaders(df: pd.DataFrame) -> tuple:
    ds_train = SRDataset(df, "train", augment=True)
    ds_val   = SRDataset(df, "val",   augment=False)
 
    dl_train = DataLoader(
        ds_train, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    )
    dl_val = DataLoader(
        ds_val, batch_size=4, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,
    )
    print(f"Train : {len(ds_train)} samples  |  {len(dl_train)} batches/epoch")
    print(f"Val   : {len(ds_val)} samples   |  {len(dl_val)} batches")
    return dl_train, dl_val, ds_train, ds_val
 
 
# ── Load split and build loaders ──────────────────────────────────────────────
df_split = pd.read_csv(SPLIT_CSV)
assert "split" in df_split.columns, "Split CSV missing 'split' column"
n_tr = (df_split["split"] == "train").sum()
n_va = (df_split["split"] == "val").sum()
print(f"Split loaded  |  train={n_tr}  val={n_va}")
 
dl_train, dl_val, ds_train, ds_val = build_sr_loaders(df_split)
 
# ── Sanity check: one batch ───────────────────────────────────────────────────
_batch = next(iter(dl_train))
_hq, _lq = _batch["hq"], _batch["lq"]
assert _hq.shape == (BATCH_SIZE, 3, PATCH_SIZE, PATCH_SIZE), \
    f"Unexpected HQ shape: {_hq.shape}"
assert _lq.shape == (BATCH_SIZE, 3, LQ_SIZE, LQ_SIZE), \
    f"Unexpected LQ shape: {_lq.shape}"
assert _hq.dtype == torch.float32 and _lq.dtype == torch.float32
assert _hq.min() >= 0.0 and _hq.max() <= 1.0
assert _lq.min() >= 0.0 and _lq.max() <= 1.0
assert not torch.isnan(_hq).any() and not torch.isnan(_lq).any()
 
print(f"[OK] Batch check passed")
print(f"     HQ shape={tuple(_hq.shape)}  range=[{_hq.min():.3f}, {_hq.max():.3f}]")
print(f"     LQ shape={tuple(_lq.shape)}  range=[{_lq.min():.3f}, {_lq.max():.3f}]")
del _batch, _hq, _lq

Split loaded  |  train=2762  val=483
Train : 2762 samples  |  345 batches/epoch
Val   : 483 samples   |  121 batches
[OK] Batch check passed
     HQ shape=(8, 3, 256, 256)  range=[0.000, 1.000]
     LQ shape=(8, 3, 64, 64)  range=[0.000, 1.000]


# Cell 6 — Loss functions
#
# Three loss components for Phase 1 (L1 + Perceptual VGG + LPIPS).
# Feature matching loss for Phase 2 is computed inside the training loop
# using the discriminator's intermediate feature maps.

In [6]:
class VGGPerceptualLoss(nn.Module):
    """
    Perceptual loss using VGG-19 feature activations at layers relu3_4 and
    relu4_4. The network is loaded once and kept frozen for the entire run.
 
    L1 distance over feature maps is used rather than L2 because it is less
    sensitive to outlier activations produced by material textures with very
    high local contrast (e.g. specular highlights on metal).
    """
 
    _LAYER_WEIGHTS = {"relu3_4": 1.0, "relu4_4": 1.0}
 
    def __init__(self):
        super().__init__()
        vgg = tvm.vgg19(weights=tvm.VGG19_Weights.IMAGENET1K_V1).features
        self.slices = nn.ModuleDict()
        slice_map = {"relu3_4": (0, 18), "relu4_4": (18, 27)}
        prev = 0
        for name, (start, end) in slice_map.items():
            self.slices[name] = nn.Sequential(*list(vgg.children())[start:end])
            prev = end
        for p in self.parameters():
            p.requires_grad = False
 
        # VGG normalisation (different from ImageNet used in MatForge encoder)
        self.register_buffer(
            "mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )
 
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # Input tensors are in [0, 1]; normalise to VGG's expected range
        pred_n   = (pred   - self.mean) / self.std
        target_n = (target - self.mean) / self.std
        loss = torch.tensor(0.0, device=pred.device)
        x_pred, x_tgt = pred_n, target_n
        for name, layer in self.slices.items():
            x_pred = layer(x_pred)
            x_tgt  = layer(x_tgt).detach()   # target features are never differentiated
            loss   = loss + self._LAYER_WEIGHTS[name] * F.l1_loss(x_pred, x_tgt)
        return loss
 
 
def feature_matching_loss(
    fake_features: list,
    real_features: list,
    weight: float = W_FM,
) -> torch.Tensor:
    """
    L1 distance between discriminator intermediate features for real and fake.
    Provides a stable gradient signal for the generator even when the
    discriminator cannot distinguish real from fake (collapse scenario).
    """
    loss = torch.tensor(0.0, device=fake_features[0].device)
    for f_fake, f_real in zip(fake_features, real_features):
        loss = loss + F.l1_loss(f_fake, f_real.detach())
    return loss * weight / len(fake_features)
 
 
def lsgan_loss_d(
    real_logits: torch.Tensor,
    fake_logits: torch.Tensor,
) -> torch.Tensor:
    """LSGAN discriminator loss. Target: real→1, fake→0."""
    return 0.5 * (F.mse_loss(real_logits, torch.ones_like(real_logits)) +
                  F.mse_loss(fake_logits, torch.zeros_like(fake_logits)))
 
 
def lsgan_loss_g(fake_logits: torch.Tensor) -> torch.Tensor:
    """LSGAN generator adversarial loss. Target: fake→1 (fool discriminator)."""
    return F.mse_loss(fake_logits, torch.ones_like(fake_logits))
 
 
def r1_gradient_penalty(
    real: torch.Tensor,
    real_logits: torch.Tensor,
) -> torch.Tensor:
    """
    R1 gradient penalty computed in float32 to avoid NaN under AMP.
    Penalises the gradient of the discriminator w.r.t. real samples.
    Computed lazily (every R1_INTERVAL steps) to reduce overhead.
    """
    with torch.amp.autocast("cuda", enabled=False):
        real_f32 = real.float().requires_grad_(True)
        logits_f32 = real_logits.float()
        grad = torch.autograd.grad(
            outputs=logits_f32.sum(),
            inputs=real_f32,
            create_graph=True,
        )[0]
    return (grad ** 2).sum([1, 2, 3]).mean() * (R1_LAMBDA * 0.5)
 
 
# ── Instantiate losses ────────────────────────────────────────────────────────
vgg_loss_fn  = VGGPerceptualLoss().to(DEVICE)
lpips_loss_fn = lpips_lib.LPIPS(net="alex").to(DEVICE)
for p in lpips_loss_fn.parameters():
    p.requires_grad = False
 
# ── Quick sanity check ────────────────────────────────────────────────────────
with torch.no_grad():
    _dummy_pred = torch.rand(2, 3, PATCH_SIZE, PATCH_SIZE, device=DEVICE)
    _dummy_gt   = torch.rand(2, 3, PATCH_SIZE, PATCH_SIZE, device=DEVICE)
    _l1   = F.l1_loss(_dummy_pred, _dummy_gt)
    _perc = vgg_loss_fn(_dummy_pred, _dummy_gt)
    # LPIPS expects input in [-1, 1]
    _lpips_val = lpips_loss_fn(
        _dummy_pred * 2 - 1, _dummy_gt * 2 - 1
    ).mean()
 
print(f"[OK] Loss functions ready")
print(f"     L1    = {_l1.item():.4f}")
print(f"     Perc  = {_perc.item():.4f}")
print(f"     LPIPS = {_lpips_val.item():.4f}")
del _dummy_pred, _dummy_gt, _l1, _perc, _lpips_val

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 192MB/s]


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 196MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
[OK] Loss functions ready
     L1    = 0.3336
     Perc  = 0.7877
     LPIPS = 0.2055


# Cell 7 — Visual validation panel
#
# Generates a panel of 6 validation samples showing LQ (bicubic upscaled),
# SR output, and HQ ground truth side by side. Saved to /kaggle/working/sr_panels/.

In [7]:
@torch.no_grad()
def generate_validation_panel(
    generator: RRDBNet,
    ds_val: Dataset,
    epoch_tag: str,
    n_samples: int = 6,
) -> None:
    """
    Renders a (n_samples × 3) grid: LQ upscaled | SR | HQ GT.
    LQ is shown at HQ resolution (bicubic) for visual alignment.
    LPIPS per sample is printed in the subplot title.
    """
    generator.eval()
    indices = list(range(0, len(ds_val), max(1, len(ds_val) // n_samples)))[:n_samples]
 
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
    fig.suptitle(f"SR Validation — {epoch_tag}", fontsize=12)
 
    for row, idx in enumerate(indices):
        sample = ds_val[idx]
        hq = sample["hq"].unsqueeze(0).to(DEVICE)
        lq = sample["lq"].unsqueeze(0).to(DEVICE)
 
        lq_up = F.interpolate(
            lq, size=(PATCH_SIZE, PATCH_SIZE),
            mode="bicubic", align_corners=False,
        ).clamp(0, 1)
 
        with torch.amp.autocast('cuda'):
            sr = generator(lq).clamp(0, 1)
 
        lpips_val = lpips_loss_fn(
            sr * 2 - 1, hq * 2 - 1
        ).item()
 
        def to_np(t):
            return t.squeeze(0).permute(1, 2, 0).cpu().float().numpy()
 
        axes[row][0].imshow(to_np(lq_up))
        axes[row][0].set_title(f"LQ bicubic  [{sample['filename']}]", fontsize=7)
        axes[row][0].axis("off")
 
        axes[row][1].imshow(to_np(sr))
        axes[row][1].set_title(f"SR output  LPIPS={lpips_val:.4f}", fontsize=7)
        axes[row][1].axis("off")
 
        axes[row][2].imshow(to_np(hq))
        axes[row][2].set_title("HQ ground truth", fontsize=7)
        axes[row][2].axis("off")
 
    plt.tight_layout()
    out_path = PANELS_DIR / f"sr_panel_{epoch_tag}.png"
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Panel saved: {out_path}")
    generator.train()

# Cell 8 — Phase 1 training loop: generator only
#
# Trains with L1 + perceptual VGG + LPIPS losses. No discriminator.
# Supports resuming from a Phase 1 checkpoint.

In [8]:
def make_discriminator_input(
    lq_up: torch.Tensor, hq: torch.Tensor
) -> torch.Tensor:
    """Concatenates upsampled LQ and HQ/SR along the channel dimension."""
    return torch.cat([lq_up, hq], dim=1)   # (B, 6, H, W)
 
 
@torch.no_grad()
def validate_phase1(
    generator: RRDBNet,
    dl_val: DataLoader,
    epoch: int,
) -> dict:
    """Computes validation metrics (L1 + LPIPS) over the full validation set."""
    generator.eval()
    total_l1, total_lpips, n = 0.0, 0.0, 0
    for batch in dl_val:
        hq = batch["hq"].to(DEVICE)
        lq = batch["lq"].to(DEVICE)
        with torch.amp.autocast('cuda'):
            sr = generator(lq).float().clamp(0, 1)
        total_l1    += F.l1_loss(sr, hq).item() * hq.size(0)
        total_lpips += lpips_loss_fn(
            sr * 2 - 1, hq * 2 - 1
        ).mean().item() * hq.size(0)
        n += hq.size(0)
    generator.train()
    return {"val_l1": total_l1 / n, "val_lpips": total_lpips / n}
 
 
def train_phase1(
    generator: RRDBNet,
    dl_train: DataLoader,
    dl_val: DataLoader,
    start_epoch: int = 0,
) -> RRDBNet:
    """
    Phase 1 training loop. Returns the generator at the best validation LPIPS.
 
    Checkpoints saved:
        sr_ft_phase1_best_lpips.pt  — best validation LPIPS (primary)
        sr_ft_phase1_best_l1.pt     — best validation L1
        sr_ft_phase1_epoch_{N}.pt   — periodic every SAVE_CKPT_EVERY epochs
        sr_ft_phase1_last.pt        — end of every epoch (for resuming)
    """
    optimizer_g = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, generator.parameters()),
        lr=LR_G_PHASE1,
        betas=(0.9, 0.99),
        weight_decay=WEIGHT_DECAY_G,
    )
    scheduler_g = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_g, T_max=PHASE1_EPOCHS, eta_min=LR_G_PHASE1 * 0.01
    )
    scaler = GradScaler('cuda')
 
    best_lpips  = float("inf")
    best_l1     = float("inf")
    patience_ct = 0
 
    # Apply initial freeze
    generator.freeze_blocks(FREEZE_BLOCKS)
    print(f"[Phase 1] First {FREEZE_BLOCKS} RRDB blocks frozen until epoch {UNFREEZE_EPOCH}")
 
    for epoch in range(start_epoch, PHASE1_EPOCHS):
        epoch_start = time.time()
 
        # Unfreeze encoder blocks at scheduled epoch
        if epoch == UNFREEZE_EPOCH:
            generator.freeze_blocks(0)
            print(f"  [Epoch {epoch}] All blocks unfrozen.")
            # Reinitialise optimiser to pick up newly unfrozen parameters
            optimizer_g = torch.optim.AdamW(
                generator.parameters(), lr=LR_G_PHASE1,
                betas=(0.9, 0.99), weight_decay=WEIGHT_DECAY_G,
            )
 
        generator.train()
        running = defaultdict(float)
        n_batches = 0
 
        for batch in dl_train:
            hq = batch["hq"].to(DEVICE, non_blocking=True)
            lq = batch["lq"].to(DEVICE, non_blocking=True)
 
            optimizer_g.zero_grad(set_to_none=True)
 
            with torch.amp.autocast('cuda'):
                sr = generator(lq)
                sr_clamp = sr.clamp(0, 1)
 
                l_l1   = F.l1_loss(sr_clamp, hq)
                l_perc = vgg_loss_fn(sr_clamp, hq)
                # LPIPS expects [-1, 1]
                l_lpips = lpips_loss_fn(
                    sr_clamp * 2 - 1, hq * 2 - 1
                ).mean()
 
                loss = W_L1 * l_l1 + W_PERC * l_perc + W_LPIPS * l_lpips
 
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer_g)
            nn.utils.clip_grad_norm_(generator.parameters(), GRAD_CLIP)
            scaler.step(optimizer_g)
            scaler.update()
 
            running["l1"]   += l_l1.item()
            running["perc"] += l_perc.item()
            running["lpips"]+= l_lpips.item()
            running["total"]+= loss.item()
            n_batches += 1
 
        scheduler_g.step()
        epoch_time = time.time() - epoch_start
        avg = {k: v / n_batches for k, v in running.items()}
 
        print(
            f"[P1 Ep {epoch:02d}/{PHASE1_EPOCHS-1}] "
            f"total={avg['total']:.4f}  l1={avg['l1']:.4f}  "
            f"perc={avg['perc']:.4f}  lpips={avg['lpips']:.4f}  "
            f"lr={optimizer_g.param_groups[0]['lr']:.2e}  "
            f"time={epoch_time:.1f}s"
        )
 
        # ── Periodic checkpoint ────────────────────────────────────────────
        if (epoch + 1) % SAVE_CKPT_EVERY == 0 or DRY_RUN:
            torch.save(generator.state_dict(),
                       CKPT_DIR / f"sr_ft_phase1_epoch_{epoch:02d}.pt")
 
        # ── Always save last ───────────────────────────────────────────────
        torch.save({"epoch": epoch, "state_dict": generator.state_dict(),
                    "optimizer": optimizer_g.state_dict()},
                   CKPT_DIR / "sr_ft_phase1_last.pt")
 
        # ── Validation ────────────────────────────────────────────────────
        if (epoch + 1) % VALIDATE_EVERY == 0 or epoch == PHASE1_EPOCHS - 1:
            metrics = validate_phase1(generator, dl_val, epoch)
            print(
                f"  [Val P1 Ep {epoch:02d}] "
                f"val_l1={metrics['val_l1']:.4f}  "
                f"val_lpips={metrics['val_lpips']:.4f}"
            )
            # Creates a validation panel
            generate_validation_panel(
                generator, ds_val,
                epoch_tag=f"phase1_ep{epoch:02d}",
            )
            # Best LPIPS checkpoint
            if metrics["val_lpips"] < best_lpips:
                best_lpips  = metrics["val_lpips"]
                patience_ct = 0
                torch.save(generator.state_dict(),
                           CKPT_DIR / "sr_ft_phase1_best_lpips.pt")
                print(f"    → New best LPIPS: {best_lpips:.4f}  (saved)")
            else:
                patience_ct += 1
                print(f"    → No LPIPS improvement  ({patience_ct}/{EARLY_STOP_PATIENCE})")
 
            # Best L1 checkpoint
            if metrics["val_l1"] < best_l1:
                best_l1 = metrics["val_l1"]
                torch.save(generator.state_dict(),
                           CKPT_DIR / "sr_ft_phase1_best_l1.pt")
 
            # Early stopping
            if patience_ct >= EARLY_STOP_PATIENCE and not DRY_RUN:
                print(f"  [Phase 1] Early stopping at epoch {epoch}.")
                break
 
    print(f"[Phase 1 complete]  Best val LPIPS={best_lpips:.4f}")
    # Load best weights for Phase 2 input
    generator.load_state_dict(
        torch.load(CKPT_DIR / "sr_ft_phase1_best_lpips.pt", map_location=DEVICE)
    )
    return generator
 
 
# ── Check for existing Phase 1 checkpoint ────────────────────────────────────
_p1_last = CKPT_DIR / "sr_ft_phase1_last.pt"
_p1_start = 0
if _p1_last.exists() and not DRY_RUN:
    _ckpt = torch.load(_p1_last, map_location=DEVICE)
    generator.load_state_dict(_ckpt["state_dict"])
    _p1_start = _ckpt["epoch"] + 1
    print(f"[Resume] Phase 1 resumed from epoch {_p1_start}")
 
generator = train_phase1(generator, dl_train, dl_val, start_epoch=_p1_start)

[Phase 1] First 2 RRDB blocks frozen until epoch 5
[P1 Ep 00/29] total=1.0343  l1=0.0353  perc=0.8404  lpips=0.3173  lr=4.99e-05  time=208.6s
[P1 Ep 01/29] total=0.9981  l1=0.0353  perc=0.8153  lpips=0.2949  lr=4.95e-05  time=213.5s
[P1 Ep 02/29] total=0.9750  l1=0.0352  perc=0.7997  lpips=0.2801  lr=4.88e-05  time=213.5s
[P1 Ep 03/29] total=0.9672  l1=0.0354  perc=0.7957  lpips=0.2721  lr=4.79e-05  time=213.2s
[P1 Ep 04/29] total=0.9611  l1=0.0355  perc=0.7924  lpips=0.2664  lr=4.67e-05  time=213.4s
  [Val P1 Ep 04] val_l1=0.0338  val_lpips=0.2667
[OK] Panel saved: /kaggle/working/sr_panels/sr_panel_phase1_ep04.png
    → New best LPIPS: 0.2667  (saved)
  [Epoch 5] All blocks unfrozen.
[P1 Ep 05/29] total=0.9589  l1=0.0356  perc=0.7919  lpips=0.2628  lr=5.00e-05  time=218.3s
[P1 Ep 06/29] total=0.9509  l1=0.0354  perc=0.7862  lpips=0.2587  lr=5.00e-05  time=218.4s
[P1 Ep 07/29] total=0.9466  l1=0.0354  perc=0.7832  lpips=0.2561  lr=5.00e-05  time=218.3s
[P1 Ep 08/29] total=0.9382  l1=0

# Cell 9 — Phase 2 training loop: generator + discriminator
#
# Adds LSGAN adversarial loss + feature matching + lazy R1 penalty.
# The discriminator receives as input the concatenation of the bicubic-upsampled
# LQ and the HQ/SR image (6 channels total). This conditioning prevents the
# discriminator from penalising correct low-frequency structure that was already
# present in the LQ input.

In [9]:
@torch.no_grad()
def validate_phase2(
    generator: RRDBNet,
    dl_val: DataLoader,
    epoch: int,
) -> dict:
    """Validation during Phase 2. Same metrics as Phase 1."""
    generator.eval()
    total_l1, total_lpips, n = 0.0, 0.0, 0
    for batch in dl_val:
        hq = batch["hq"].to(DEVICE)
        lq = batch["lq"].to(DEVICE)
        with torch.amp.autocast('cuda'):
            sr = generator(lq).float().clamp(0, 1)
        total_l1    += F.l1_loss(sr, hq).item() * hq.size(0)
        total_lpips += lpips_loss_fn(
            sr * 2 - 1, hq * 2 - 1
        ).mean().item() * hq.size(0)
        n += hq.size(0)
    generator.train()
    return {"val_l1": total_l1 / n, "val_lpips": total_lpips / n}
 
 
def train_phase2(
    generator: RRDBNet,
    dl_train: DataLoader,
    dl_val: DataLoader,
) -> RRDBNet:
    """
    Phase 2 training loop with GAN fine-tuning.
    Returns the generator at the best validation LPIPS across Phase 2.
 
    Collapse detection: if D(real) and D(fake) both converge to ~0.5 for
    3 consecutive epochs, Phase 2 is aborted and the Phase 1 best checkpoint
    is restored. This mirrors the GAN fine-tuning behaviour observed in MatForge.
 
    Checkpoints saved:
        sr_ft_phase2_best_lpips.pt  — best validation LPIPS (primary)
        sr_ft_phase2_epoch_{N}.pt   — periodic
        sr_ft_phase2_last.pt        — resumable state
    """
    discriminator = build_discriminator()
 
    optimizer_g = torch.optim.AdamW(
        generator.parameters(), lr=LR_G_PHASE2,
        betas=(0.9, 0.99), weight_decay=WEIGHT_DECAY_G,
    )
    optimizer_d = torch.optim.AdamW(
        discriminator.parameters(), lr=LR_D,
        betas=(0.0, 0.99), weight_decay=WEIGHT_DECAY_D,
    )
    scaler_g = GradScaler('cuda')
    scaler_d = GradScaler('cuda')
 
    best_lpips   = float("inf")
    patience_ct  = 0
    collapse_ct  = 0
    d_step_total = 0
 
    for epoch in range(PHASE2_EPOCHS):
        epoch_start = time.time()
        w_adv = get_adv_weight(epoch)
        running = defaultdict(float)
        n_batches = 0
 
        generator.train()
        discriminator.train()
 
        for batch in dl_train:
            hq = batch["hq"].to(DEVICE, non_blocking=True)
            lq = batch["lq"].to(DEVICE, non_blocking=True)
 
            # Bicubic upscale for discriminator conditioning input
            lq_up = F.interpolate(
                lq, size=(PATCH_SIZE, PATCH_SIZE),
                mode="bicubic", align_corners=False,
            ).clamp(0, 1).detach()
 
            # ── Discriminator update ───────────────────────────────────────
            optimizer_d.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                sr = generator(lq).clamp(0, 1).detach()
                real_input = make_discriminator_input(lq_up, hq)
                fake_input = make_discriminator_input(lq_up, sr)
                real_logits, _ = discriminator(real_input)
                fake_logits, _ = discriminator(fake_input)
                l_d = lsgan_loss_d(real_logits, fake_logits)

            scaler_d.scale(l_d).backward()

            # Lazy R1: separate backward to avoid mixed-dtype graph
            if d_step_total % R1_INTERVAL == 0:
                optimizer_d.zero_grad(set_to_none=True)
                real_input_r1 = make_discriminator_input(lq_up, hq).requires_grad_(True)
                real_logits_r1, _ = discriminator(real_input_r1)
                r1 = r1_gradient_penalty(real_input_r1, real_logits_r1)
                scaler_d.scale(r1).backward()

            scaler_d.unscale_(optimizer_d)
            nn.utils.clip_grad_norm_(discriminator.parameters(), GRAD_CLIP)
            scaler_d.step(optimizer_d)
            scaler_d.update()
            d_step_total += 1
 
            # ── Generator update ───────────────────────────────────────────
            optimizer_g.zero_grad(set_to_none=True)
 
            with torch.amp.autocast('cuda'):
                sr = generator(lq).clamp(0, 1)
                fake_input_g = make_discriminator_input(lq_up, sr)
                real_input_g = make_discriminator_input(lq_up, hq)
 
                fake_logits_g, fake_feats = discriminator(fake_input_g)
                with torch.no_grad():
                    _, real_feats = discriminator(real_input_g)
 
                l_l1   = F.l1_loss(sr, hq)
                l_perc = vgg_loss_fn(sr, hq)
                l_lpips = lpips_loss_fn(sr * 2 - 1, hq * 2 - 1).mean()
                l_adv  = lsgan_loss_g(fake_logits_g)
                l_fm   = feature_matching_loss(fake_feats, real_feats)
 
                loss_g = (W_L1  * l_l1  + W_PERC * l_perc +
                          W_LPIPS * l_lpips + w_adv * l_adv + l_fm)
 
            scaler_g.scale(loss_g).backward()
            scaler_g.unscale_(optimizer_g)
            nn.utils.clip_grad_norm_(generator.parameters(), GRAD_CLIP)
            scaler_g.step(optimizer_g)
            scaler_g.update()
 
            running["l_d"]    += l_d.item()
            running["l_adv"]  += l_adv.item()
            running["l_fm"]   += l_fm.item()
            running["l_l1"]   += l_l1.item()
            running["l_perc"] += l_perc.item()
            running["l_lpips"]+= l_lpips.item()
            running["d_real"] += real_logits.mean().item()
            running["d_fake"] += fake_logits.mean().item()
            n_batches += 1
 
        epoch_time = time.time() - epoch_start
        avg = {k: v / n_batches for k, v in running.items()}
 
        print(
            f"[P2 Ep {epoch:02d}/{PHASE2_EPOCHS-1}] "
            f"D={avg['l_d']:.3f}  G_adv={avg['l_adv']:.3f}  "
            f"FM={avg['l_fm']:.3f}  L1={avg['l_l1']:.4f}  "
            f"Perc={avg['l_perc']:.4f}  LPIPS={avg['l_lpips']:.4f}  "
            f"D(real)={avg['d_real']:.3f}  D(fake)={avg['d_fake']:.3f}  "
            f"w_adv={w_adv}  time={epoch_time:.1f}s"
        )
 
        # ── Discriminator collapse detection ──────────────────────────────
        d_real_avg = avg["d_real"]
        d_fake_avg = avg["d_fake"]
        if abs(d_real_avg - 0.5) < 0.05 and abs(d_fake_avg - 0.5) < 0.05:
            collapse_ct += 1
            print(f"  [WARN] Discriminator collapse signal ({collapse_ct}/3): "
                  f"D(real)={d_real_avg:.3f}  D(fake)={d_fake_avg:.3f}")
            if collapse_ct >= 3 and not DRY_RUN:
                print("  [Phase 2] Discriminator collapsed — aborting Phase 2.")
                print("  Restoring best Phase 1 checkpoint.")
                p1_best = CKPT_DIR / "sr_ft_phase1_best_lpips.pt"
                if p1_best.exists():
                    generator.load_state_dict(
                        torch.load(p1_best, map_location=DEVICE)
                    )
                return generator
        else:
            collapse_ct = 0
 
        # ── Periodic checkpoint ────────────────────────────────────────────
        if (epoch + 1) % SAVE_CKPT_EVERY == 0 or DRY_RUN:
            torch.save(generator.state_dict(),
                       CKPT_DIR / f"sr_ft_phase2_epoch_{epoch:02d}.pt")
 
        torch.save({
            "epoch": epoch,
            "generator": generator.state_dict(),
            "discriminator": discriminator.state_dict(),
            "opt_g": optimizer_g.state_dict(),
            "opt_d": optimizer_d.state_dict(),
        }, CKPT_DIR / "sr_ft_phase2_last.pt")
 
        # ── Validation ────────────────────────────────────────────────────
        if (epoch + 1) % VALIDATE_EVERY == 0 or epoch == PHASE2_EPOCHS - 1:
            metrics = validate_phase2(generator, dl_val, epoch)
            print(
                f"  [Val P2 Ep {epoch:02d}] "
                f"val_l1={metrics['val_l1']:.4f}  "
                f"val_lpips={metrics['val_lpips']:.4f}"
            )
            generate_validation_panel(
                generator, ds_val,
                epoch_tag=f"phase2_ep{epoch:02d}",
            )
            if metrics["val_lpips"] < best_lpips:
                best_lpips  = metrics["val_lpips"]
                patience_ct = 0
                torch.save(generator.state_dict(),
                           CKPT_DIR / "sr_ft_phase2_best_lpips.pt")
                print(f"    → New best LPIPS (P2): {best_lpips:.4f}  (saved)")
            else:
                patience_ct += 1
                if patience_ct >= EARLY_STOP_PATIENCE and not DRY_RUN:
                    print(f"  [Phase 2] Early stopping at epoch {epoch}.")
                    break
 
    print(f"[Phase 2 complete]  Best val LPIPS={best_lpips:.4f}")
    p2_best = CKPT_DIR / "sr_ft_phase2_best_lpips.pt"
    if p2_best.exists():
        generator.load_state_dict(torch.load(p2_best, map_location=DEVICE))
    return generator

_p2_last = CKPT_DIR / "sr_ft_phase2_last.pt"
if _p2_last.exists() and not DRY_RUN:
    _p2_ckpt = torch.load(_p2_last, map_location=DEVICE)
    generator.load_state_dict(_p2_ckpt["generator"])
    print(f"[Resume] Phase 2 state found — but train_phase2 restarts from epoch 0 "
          f"with loaded generator weights (Phase 2 resume not fully implemented).")
 
 
generator = train_phase2(generator, dl_train, dl_val)
generate_validation_panel(generator, ds_val, epoch_tag="final")
print("[Done] All cells completed successfully.")
print(f"Checkpoints in: {CKPT_DIR}")
print(f"Panels in     : {PANELS_DIR}")

[OK] Discriminator built  |  params=4.75M
[P2 Ep 00/19] D=0.260  G_adv=0.273  FM=0.015  L1=0.0345  Perc=0.7586  LPIPS=0.2251  D(real)=0.487  D(fake)=0.486  w_adv=0.01  time=426.8s
  [WARN] Discriminator collapse signal (1/3): D(real)=0.487  D(fake)=0.486
[P2 Ep 01/19] D=0.250  G_adv=0.255  FM=0.012  L1=0.0344  Perc=0.7594  LPIPS=0.2244  D(real)=0.497  D(fake)=0.496  w_adv=0.01  time=424.5s
  [WARN] Discriminator collapse signal (2/3): D(real)=0.497  D(fake)=0.496
[P2 Ep 02/19] D=0.250  G_adv=0.253  FM=0.010  L1=0.0343  Perc=0.7562  LPIPS=0.2226  D(real)=0.498  D(fake)=0.497  w_adv=0.01  time=423.7s
  [WARN] Discriminator collapse signal (3/3): D(real)=0.498  D(fake)=0.497
  [Phase 2] Discriminator collapsed — aborting Phase 2.
  Restoring best Phase 1 checkpoint.
[OK] Panel saved: /kaggle/working/sr_panels/sr_panel_final.png
[Done] All cells completed successfully.
Checkpoints in: /kaggle/working/sr_checkpoints
Panels in     : /kaggle/working/sr_panels
